In [37]:
import pandas as pd
import numpy as np

df = pd.read_csv("../dataset/processed/cleaned_tickets.csv")

print("Shape:", df.shape)
print(df.columns.tolist())

Shape: (20000, 17)
['subject', 'body', 'answer', 'type', 'queue', 'priority', 'language', 'tag_1', 'tag_2', 'tag_3', 'tag_4', 'tag_5', 'tag_6', 'tag_7', 'tag_8', 'subject_length', 'body_length']


In [38]:
print(df["priority"].value_counts())
print("\nPercentage:")
print(df["priority"].value_counts(normalize=True) * 100)

priority
medium    8144
high      7801
low       4055
Name: count, dtype: int64

Percentage:
priority
medium    40.720
high      39.005
low       20.275
Name: proportion, dtype: float64


In [39]:
print(df.isnull().sum())

subject           1461
body                 2
answer               4
type                 0
queue                0
priority             0
language             0
tag_1                0
tag_2                0
tag_3                0
tag_4                0
tag_5                0
tag_6                0
tag_7                0
tag_8                0
subject_length       0
body_length          0
dtype: int64


In [40]:
df["subject"] = df["subject"].fillna("")
df["body"] = df["body"].fillna("")

df["ticket_text"] = (
    df["subject"] + " " + df["body"]
)

print("Missing subject:", df["subject"].isna().sum())
print("Missing body:", df["body"].isna().sum())

Missing subject: 0
Missing body: 0


In [41]:
print("Duplicate ticket texts:", df["ticket_text"].duplicated().sum())

Duplicate ticket texts: 0


In [43]:
df["subject"] = df["subject"].fillna("")
df["body"] = df["body"].fillna("")

df["ticket_text"] = (
    df["subject"] + " " + df["body"]
)

print(df["ticket_text"].head())

0    Unvorhergesehener Absturz der Datenanalyse-Pla...
1    Customer Support Inquiry Seeking information o...
2    Data Analytics for Investment I am contacting ...
3    Krankenhaus-Dienstleistung-Problem Ein Medien-...
4    Security Dear Customer Support, I am reaching ...
Name: ticket_text, dtype: str


In [46]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["priority"])
y = df["priority"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (16000, 17)
Test: (4000, 17)


In [47]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
    strip_accents="unicode"
)

X_train_text = tfidf.fit_transform(X_train["ticket_text"])
X_test_text = tfidf.transform(X_test["ticket_text"])

print("Training shape:", X_train_text.shape)
print("Testing shape:", X_test_text.shape)

Training shape: (16000, 20000)
Testing shape: (4000, 20000)


In [48]:
from sklearn.svm import LinearSVC

baseline_model = LinearSVC(
    C=1.0,
    class_weight="balanced",
    max_iter=10000
)

baseline_model.fit(X_train_text, y_train)

y_pred = baseline_model.predict(X_test_text)

print("Model trained successfully!")

Model trained successfully!


In [49]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

accuracy = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")

print("Baseline Accuracy:", accuracy)
print("Baseline Macro F1:", macro_f1)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Baseline Accuracy: 0.5365
Baseline Macro F1: 0.519389521262067

Classification Report:
              precision    recall  f1-score   support

        high       0.57      0.58      0.58      1560
         low       0.44      0.43      0.43       811
      medium       0.55      0.55      0.55      1629

    accuracy                           0.54      4000
   macro avg       0.52      0.52      0.52      4000
weighted avg       0.54      0.54      0.54      4000



In [50]:
from scipy.sparse import hstack
from sklearn.preprocessing import OneHotEncoder

encoder_type = OneHotEncoder(handle_unknown="ignore")

X_train_type = encoder_type.fit_transform(
    X_train[["type"]]
)

X_test_type = encoder_type.transform(
    X_test[["type"]]
)

X_train_type_combined = hstack([
    X_train_text,
    X_train_type
])

X_test_type_combined = hstack([
    X_test_text,
    X_test_type
])

print("New training shape:", X_train_type_combined.shape)
print("New testing shape:", X_test_type_combined.shape)

New training shape: (16000, 20004)
New testing shape: (4000, 20004)


In [51]:
model_type = LinearSVC(
    C=1.0,
    class_weight="balanced",
    max_iter=10000
)

model_type.fit(X_train_type_combined, y_train)

y_pred_type = model_type.predict(X_test_type_combined)

In [52]:
print("Accuracy:",
      accuracy_score(y_test, y_pred_type))

print("Macro F1:",
      f1_score(y_test, y_pred_type, average="macro"))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_type))

Accuracy: 0.53425
Macro F1: 0.5163419312240514

Classification Report:
              precision    recall  f1-score   support

        high       0.57      0.58      0.57      1560
         low       0.44      0.42      0.43       811
      medium       0.55      0.55      0.55      1629

    accuracy                           0.53      4000
   macro avg       0.52      0.52      0.52      4000
weighted avg       0.53      0.53      0.53      4000



In [53]:
encoder_queue = OneHotEncoder(handle_unknown="ignore")

X_train_queue = encoder_queue.fit_transform(
    X_train[["queue"]]
)

X_test_queue = encoder_queue.transform(
    X_test[["queue"]]
)

X_train_queue_combined = hstack([
    X_train_text,
    X_train_queue
])

X_test_queue_combined = hstack([
    X_test_text,
    X_test_queue
])

print("Training shape:", X_train_queue_combined.shape)
print("Testing shape:", X_test_queue_combined.shape)

Training shape: (16000, 20010)
Testing shape: (4000, 20010)


In [54]:
model_queue = LinearSVC(
    C=1.0,
    class_weight="balanced",
    max_iter=10000
)

model_queue.fit(X_train_queue_combined, y_train)

y_pred_queue = model_queue.predict(X_test_queue_combined)

In [55]:
print("Accuracy:",
      accuracy_score(y_test, y_pred_queue))

print("Macro F1:",
      f1_score(y_test, y_pred_queue, average="macro"))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_queue))

Accuracy: 0.58325
Macro F1: 0.5661552959191318

Classification Report:
              precision    recall  f1-score   support

        high       0.62      0.65      0.63      1560
         low       0.50      0.47      0.48       811
      medium       0.59      0.58      0.58      1629

    accuracy                           0.58      4000
   macro avg       0.57      0.56      0.57      4000
weighted avg       0.58      0.58      0.58      4000



In [56]:
encoder_language = OneHotEncoder(handle_unknown="ignore")

X_train_language = encoder_language.fit_transform(
    X_train[["language"]]
)

X_test_language = encoder_language.transform(
    X_test[["language"]]
)

X_train_queue_language = hstack([
    X_train_text,
    X_train_queue,
    X_train_language
])

X_test_queue_language = hstack([
    X_test_text,
    X_test_queue,
    X_test_language
])

print("Training shape:", X_train_queue_language.shape)
print("Testing shape:", X_test_queue_language.shape)

Training shape: (16000, 20012)
Testing shape: (4000, 20012)


In [57]:
model_queue_language = LinearSVC(
    C=1.0,
    class_weight="balanced",
    max_iter=10000
)

model_queue_language.fit(
    X_train_queue_language,
    y_train
)

y_pred_queue_language = model_queue_language.predict(
    X_test_queue_language
)

In [58]:
print("Accuracy:",
      accuracy_score(y_test, y_pred_queue_language))

print("Macro F1:",
      f1_score(
          y_test,
          y_pred_queue_language,
          average="macro"
      ))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred_queue_language
    )
)

Accuracy: 0.5835
Macro F1: 0.5664949390868668

Classification Report:
              precision    recall  f1-score   support

        high       0.62      0.65      0.63      1560
         low       0.50      0.47      0.48       811
      medium       0.59      0.58      0.58      1629

    accuracy                           0.58      4000
   macro avg       0.57      0.57      0.57      4000
weighted avg       0.58      0.58      0.58      4000



In [59]:

tag_cols = [
    "tag_1", "tag_2", "tag_3", "tag_4",
    "tag_5", "tag_6", "tag_7", "tag_8"
]

encoder_tags = OneHotEncoder(
    handle_unknown="ignore"
)

X_train_tags = encoder_tags.fit_transform(
    X_train[tag_cols]
)

X_test_tags = encoder_tags.transform(
    X_test[tag_cols]
)

X_train_all_tags = hstack([
    X_train_text,
    X_train_queue,
    X_train_language,
    X_train_tags
])

X_test_all_tags = hstack([
    X_test_text,
    X_test_queue,
    X_test_language,
    X_test_tags
])

print("Training shape:", X_train_all_tags.shape)
print("Testing shape:", X_test_all_tags.shape)

Training shape: (16000, 22884)
Testing shape: (4000, 22884)


In [60]:
model_tags = LinearSVC(
    C=1.0,
    class_weight="balanced",
    max_iter=10000
)

model_tags.fit(
    X_train_all_tags,
    y_train
)

y_pred_tags = model_tags.predict(
    X_test_all_tags
)

In [61]:
print("Accuracy:",
      accuracy_score(y_test, y_pred_tags))

print("Macro F1:",
      f1_score(
          y_test,
          y_pred_tags,
          average="macro"
      ))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred_tags
    )
)

Accuracy: 0.56825
Macro F1: 0.5510024258353708

Classification Report:
              precision    recall  f1-score   support

        high       0.61      0.65      0.63      1560
         low       0.48      0.45      0.47       811
      medium       0.57      0.55      0.56      1629

    accuracy                           0.57      4000
   macro avg       0.55      0.55      0.55      4000
weighted avg       0.57      0.57      0.57      4000



In [1]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, classification_report
from scipy.sparse import hstack

# ---- Config: flip this depending on what you're optimizing for ----
USE_CLASS_WEIGHT = False   # True -> better macro F1 / recall on minority classes
                           # False -> best raw accuracy (58.95%)
C_VALUE = 1.0

# ---- Load & prep ----
df = pd.read_csv("../dataset/processed/cleaned_tickets.csv")
df["subject"] = df["subject"].fillna("")
df["body"] = df["body"].fillna("")
df["ticket_text"] = df["subject"] + " " + df["body"]

X = df.drop(columns=["priority"])
y = df["priority"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# ---- Feature engineering: text + queue + type (best-performing combo) ----
tfidf = TfidfVectorizer(
    max_features=30000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
    strip_accents="unicode",
)
X_train_text = tfidf.fit_transform(X_train["ticket_text"])
X_test_text = tfidf.transform(X_test["ticket_text"])

encoder_cat = OneHotEncoder(handle_unknown="ignore")
X_train_cat = encoder_cat.fit_transform(X_train[["queue", "type"]])
X_test_cat = encoder_cat.transform(X_test[["queue", "type"]])

X_train_final = hstack([X_train_text, X_train_cat]).tocsr()
X_test_final = hstack([X_test_text, X_test_cat]).tocsr()

# ---- Train ----
model = LinearSVC(
    C=C_VALUE,
    class_weight="balanced" if USE_CLASS_WEIGHT else None,
    max_iter=10000,
)
model.fit(X_train_final, y_train)

# ---- Evaluate ----
y_pred = model.predict(X_test_final)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Macro F1:", f1_score(y_test, y_pred, average="macro"))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))



Accuracy: 0.5885
Macro F1: 0.5585144267420779

Classification Report:
              precision    recall  f1-score   support

        high       0.62      0.67      0.64      1560
         low       0.54      0.36      0.44       811
      medium       0.58      0.62      0.60      1629

    accuracy                           0.59      4000
   macro avg       0.58      0.55      0.56      4000
weighted avg       0.58      0.59      0.58      4000



In [2]:
import os
import joblib
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, classification_report
from scipy.sparse import hstack

# -----------------------------
# Load data
# -----------------------------

df = pd.read_csv("../dataset/processed/cleaned_tickets.csv")

df["subject"] = df["subject"].fillna("")
df["body"] = df["body"].fillna("")

df["ticket_text"] = df["subject"] + " " + df["body"]

X = df.drop(columns=["priority"])
y = df["priority"]

# -----------------------------
# Train-test split
# -----------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# -----------------------------
# TF-IDF
# -----------------------------

tfidf = TfidfVectorizer(
    max_features=30000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
    strip_accents="unicode"
)

X_train_text = tfidf.fit_transform(X_train["ticket_text"])
X_test_text = tfidf.transform(X_test["ticket_text"])

# -----------------------------
# Queue + Type
# -----------------------------

encoder_cat = OneHotEncoder(
    handle_unknown="ignore"
)

X_train_cat = encoder_cat.fit_transform(
    X_train[["queue", "type"]]
)

X_test_cat = encoder_cat.transform(
    X_test[["queue", "type"]]
)

# -----------------------------
# Combine features
# -----------------------------

X_train_final = hstack([
    X_train_text,
    X_train_cat
]).tocsr()

X_test_final = hstack([
    X_test_text,
    X_test_cat
]).tocsr()

print("Training shape:", X_train_final.shape)
print("Testing shape:", X_test_final.shape)

# -----------------------------
# Train model
# -----------------------------

model = LinearSVC(
    C=1.0,
    class_weight=None,
    max_iter=10000
)

model.fit(X_train_final, y_train)

# -----------------------------
# Evaluate
# -----------------------------

y_pred = model.predict(X_test_final)

accuracy = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(
    y_test,
    y_pred,
    average="macro"
)

print("\nAccuracy:", accuracy)
print("Macro F1:", macro_f1)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# -----------------------------
# Save model + preprocessing
# -----------------------------

os.makedirs("../models/priority", exist_ok=True)

joblib.dump(
    model,
    "../models/priority/priority_model.pkl"
)

joblib.dump(
    tfidf,
    "../models/priority/tfidf_vectorizer.pkl"
)

joblib.dump(
    encoder_cat,
    "../models/priority/category_encoder.pkl"
)

print("\n✅ Best priority model saved successfully!")

Training shape: (16000, 30014)
Testing shape: (4000, 30014)

Accuracy: 0.5885
Macro F1: 0.5585144267420779

Classification Report:
              precision    recall  f1-score   support

        high       0.62      0.67      0.64      1560
         low       0.54      0.36      0.44       811
      medium       0.58      0.62      0.60      1629

    accuracy                           0.59      4000
   macro avg       0.58      0.55      0.56      4000
weighted avg       0.58      0.59      0.58      4000


✅ Best priority model saved successfully!
